# DCT Laboratory — Volume I, Chapter 13
## Unified Enterprise Transformation Architecture
**Seed `26113`** · Companion to Chapter 13 and **AXIOM Lab 1.13** (module **AXIOM-13**) · Mirrored in `DCT_V1_Ch13_Lab.xlsx`

This laboratory reproduces **Worked Example 13.1, *Is the Same Cash Promised Twice?***, with the book's
declarations, and works the five steps of AXIOM Lab 1.13, *One Company, Five Ledgers*: assemble, wire the
couplings, follow the wavefront, audit the cycle, decide in one place. The breach odds the worked example
cites are Worked Example 8.1's 100,000 paths, drawn as in the book with **random seed 8** (Volume II, Online
Appendix 1.B.4), so they come out exactly as printed; the lab's seed 26113 is used only for fresh draws,
labelled as such. Every Meridian number below is the book's or follows from its declarations (Worked Example
13.1, Tables 13.1 and 13.6, Figures 13.1–13.3, Online Appendix 13.B.1–13.B.4); the final cell checks the
printed ones. Money is in \$ million.

## The declarations (Worked Example 13.1, Step 1; Table 13.1)

**The skeleton** (Chapter 9, Worked Example 9.1). Seven components: the divisions Industrial Systems (IND),
Digital Solutions (DIG) and Advanced Materials (MAT), shared engineering (ENG), treasury (TRS), the market
(MKT) and governance (GOV). Fourteen typed arrows wire them: demand from the market to each division, cash
from the divisions to treasury, funding from treasury to Industrial Systems and Digital, capability both ways
between engineering and those two divisions, and policy from governance to treasury and Digital. The
constraints (Table 9.3): the covenant (net leverage at most 4.5 times) and the \$350 million liquidity floor
at treasury, plant capacity at Industrial Systems, the strategic-risk ceiling of 75 across the three
divisions, and workforce capability of at least 55 for the digital build across engineering and Digital.

**The layers on it** (Table 13.1). *Capital:* the ten forms of Table 10.1; treasury carries liquidity and net
leverage, engineering the workforce-capability index (Table 9.1). *Performance:* the ten dimensions of
Table 11.2 with the supports of Table 11.4, and the CFO's scorecard $(L + T + E - R)/4$ on Chapter 1's
readings, 37.0 today. *Risk:* the loss channels of Table 12.2 with their supports, the walls, and the risk
office's closure buffer. *Transformation:* the digital build, which writes to Digital and engineering
(Online Appendix 13.B.3). An indicator read by two layers carries its one Table 5.2 contract: liquidity from
treasury systems (±1 percent), strategic risk from the risk committee (±6 points).

**The digital program** (Chapters 1 and 8). Liquidity today is \$620 million, \$6.2 million an index point.
Chapter 1's digital path (Online Appendix 1.B.4) relaxes toward 112 at 35 percent a year, less a trough of
depth 28 reached after two years (the phased build: depth 20); Chapter 8's plan reads it quarterly over the
program's three years. The board's floor is \$350 million, and its appetite tolerates at most a 5 percent
chance of breaching the floor during the program.

**The uncertainty** (Chapters 8 and 12). Each quarter brings an execution surprise with a standard deviation
of \$50 million, 60 percent of which carries into the next quarter. A funding window closes about once a
decade (0.10 a year) and removes a gamma-distributed amount, mean \$120 million and standard deviation
\$50 million, that is not recovered within the program. The standby line covers \$100 million of any closure.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 26113          # the lab's seed: fresh draws only, labelled as such
BOOK_SEED = 8         # the book's stream for Worked Example 8.1's paths (Volume II, Online Appendix 1.B.4)

# The skeleton: Meridian's state architecture (Chapter 9, Worked Example 9.1; Tables 9.1 and 9.3)
COMPONENTS = ["MKT", "GOV", "MAT", "IND", "DIG", "ENG", "TRS"]
NAMES = {"MKT": "the market", "GOV": "governance", "MAT": "Advanced Materials", "IND": "Industrial Systems",
         "DIG": "Digital", "ENG": "engineering", "TRS": "treasury"}
ARROWS = [("MKT", "IND", "demand"), ("MKT", "DIG", "demand"), ("MKT", "MAT", "demand"),
          ("IND", "TRS", "cash"), ("DIG", "TRS", "cash"), ("MAT", "TRS", "cash"),
          ("TRS", "IND", "funding"), ("TRS", "DIG", "funding"),
          ("ENG", "IND", "capability"), ("IND", "ENG", "capability"),
          ("ENG", "DIG", "capability"), ("DIG", "ENG", "capability"),
          ("GOV", "TRS", "policy"), ("GOV", "DIG", "policy")]
CONSTRAINTS = {"covenant: net leverage at most 4.5x": ["TRS"], "liquidity floor of $350 million": ["TRS"],
               "plant capacity": ["IND"], "strategic-risk ceiling of 75": ["IND", "DIG", "MAT"],
               "capability of at least 55 for the digital build": ["ENG", "DIG"]}
# Table 5.2: one measurement contract per indicator
CONTRACTS = {"liquidity": "treasury systems, +/-1%", "net leverage": "audited accounts, +/-2%",
             "workforce capability": "skills survey, +/-5 points", "technology platform": "architecture audit, +/-4 points",
             "operational efficiency": "cost benchmarking, +/-3 points", "strategic risk": "risk committee, +/-6 points"}
FORMS = ["financial", "human", "intellectual", "technological", "organizational", "relational",
         "strategic", "governance", "innovation", "informational"]                      # Table 10.1
DIMENSIONS = ["financial", "operational", "strategic", "human", "technological", "innovation",
              "governance", "environmental", "resilience", "sustainability"]            # Table 11.2

CAPABILITY, CAPABILITY_FLOOR = 58.0, 55.0            # workforce capability today (Table 3.1); the build's floor

def reach(sources):
    # earliest quarter at which a change at the sources can reach each component (None: never)
    dist, frontier, k = {s: 0 for s in sources}, list(sources), 0
    while frontier:
        k, nxt = k + 1, []
        for u in frontier:
            for a, b, _ in ARROWS:
                if a == u and b not in dist:
                    dist[b] = k
                    nxt.append(b)
        frontier = nxt
    return {c: dist.get(c) for c in COMPONENTS}

# The digital program (Chapter 1's model, Online Appendix 1.B.4; Chapter 8's plan), $ million
LIQ0, FLOOR, PER_POINT, APPETITE = 620.0, 350.0, 6.2, 0.05
POOL = LIQ0 - FLOOR                                          # cash above the floor

def relax(x0, target, k, t):
    return target + (x0 - target) * np.exp(-k * t)

def trough(depth, tau, t):
    return depth * (t / tau) * np.exp(1 - t / tau)

def digital(t, depth=28.0):
    # the four moving readings: liquidity index, technology platform, efficiency, strategic risk (t in years)
    return {"L": relax(100.0, 112.0, 0.35, t) - trough(depth, 2.0, t), "T": relax(41.0, 78.0, 0.55, t),
            "E": relax(71.0, 77.0, 0.50, t) - trough(9.0, 1.5, t), "R": relax(64.0, 51.0, 0.40, t) + trough(14.0, 1.2, t)}

def score(r):
    return (r["L"] + r["T"] + r["E"] - r["R"]) / 4             # the CFO's scorecard (Worked Example 1.1; Chapter 11)

def plan(depth=28.0):
    return PER_POINT * digital(np.arange(13) / 4, depth)["L"]   # Chapter 8's plan, quarters 0-12

# The uncertainty (Chapter 8, Worked Example 8.1; Chapter 12's closures) and the levers
A_E, SD_W = 0.6, 50.0                                      # execution deviation: share carried, surprise sd
LAM, J_MEAN, J_SD = 0.10, 120.0, 50.0                      # closures a year; size mean and sd
SHAPE, SCALE = (J_MEAN / J_SD) ** 2, J_SD ** 2 / J_MEAN    # gamma sizes with that mean and sd
LINE, DEPTH, DEPTH_PHASED = 100.0, 28.0, 20.0              # the standby line per closure; the draw's depth

def simulate(line=0.0, depth=DEPTH, n=100_000, seed=BOOK_SEED):
    # Liquidity paths [n, 13] in the book engine's order (we_8.py): each quarter the n surprises, the n closure
    # counts, then the sizes of that quarter's closures in path order; seed 8 gives the book's paths exactly.
    rng = np.random.default_rng(seed)
    pl, E, J = plan(depth), np.zeros(n), np.zeros(n)
    L = np.empty((n, 13)); L[:, 0] = pl[0]
    for k in range(12):
        w = SD_W * rng.standard_normal(n)
        cnt = rng.poisson(LAM / 4, n)
        sizes = rng.gamma(SHAPE, SCALE, cnt.sum())
        lost = np.bincount(np.repeat(np.arange(n), cnt), np.maximum(sizes - line, 0.0), minlength=n)
        E, J = A_E * E + w, J - lost
        L[:, k + 1] = pl[k + 1] + E + J
    return L

print(f"skeleton: {len(COMPONENTS)} components, {len(ARROWS)} arrows, {len(CONSTRAINTS)} constraints")
print(f"liquidity today ${LIQ0:.0f} million, floor ${FLOOR:.0f} million: ${POOL:.0f} million above it")
print(f"closure sizes: gamma, shape {SHAPE:.2f} and scale {SCALE:.2f} (mean {SHAPE * SCALE:.0f}, sd {np.sqrt(SHAPE) * SCALE:.0f})")
print(f"the CFO's scorecard today: ({100:.0f} + {41:.0f} + {71:.0f} - {64:.0f})/4 = {score(digital(0.0)):.1f}")

## Panel 1 — Assemble (Theorems 13.1 and 13.2(i); Definition 13.4; Table 13.1: seven components, a feedback core of four, ten forms, ten dimensions)

AXIOM Lab 1.13, step 1. Attach the four layers to Meridian's skeleton in two different orders and read the
compatibility checks at each attachment: **(C1)** every carrier, support, write set and buffer is a component;
**(C2)** an indicator read by two layers carries one measurement contract; **(C3)** every risk buffer is named
to a declared capital stock, its funding identity; **(C4)** the layers' constraints have a joint solution,
which for Meridian is the joint check of the claims on treasury cash (Table 13.3); **(C5)** operator domains
are recomputed against it, the digital build's among them. Integration adjoins a declaration and changes nothing
already attached (Definition 13.3), so the unified object is the skeleton with the *set* of declarations:
both orders give the same one (Theorem 13.2(i)), and restricting it to one layer returns that layer
unchanged (Theorem 13.1(iii)). The order changes only *when* (C4) fails, never *whether*: it fails at the
attachment that puts the program's draw and the closure buffer on the same treasury cash, whichever comes
second. No layer can be dropped (Theorem 13.1(iv)): two risk layers that differ only in their measure
answer the same question differently.

In [ ]:
DRAW = LIQ0 - plan().min()                                     # the program's deepest draw (quarter 7)
BUFFER = stats.gamma.ppf(0.90, SHAPE, scale=SCALE)             # the 90th-percentile closure (Chapter 12)
LAYERS = {
    "transformation": {"refs": {"digital build: write set": ["DIG", "ENG"], "program draw D": ["DIG"]},
                       "claims": {"program draw D": DRAW}},
    "capital": {"refs": {"liquidity": ["TRS"], "net leverage": ["TRS"], "workforce capability": ["ENG"],
                         "cash above the floor A": ["TRS"]},
                "reads": {k: CONTRACTS[k] for k in ("liquidity", "net leverage", "workforce capability")},
                "stocks": {"cash above the floor A": POOL}, "forms": FORMS},
    "performance": {"refs": {"P_F": ["TRS"], "P_O": ["IND", "MAT"], "P_S": ["IND", "DIG", "MAT"], "P_T": ["DIG"],
                             "P_I": ["DIG", "ENG"], "P_R": ["TRS"]},                       # Table 11.4
                    "reads": {k: CONTRACTS[k] for k in ("liquidity", "technology platform", "operational efficiency",
                                                        "strategic risk")},             # the CFO's scorecard
                    "dimensions": DIMENSIONS},
    "risk": {"refs": {"R_F": ["TRS"], "R_O": ["IND", "MAT"], "R_S": ["MKT", "GOV"], "R_T": ["DIG", "ENG"],
                      "R_G": ["GOV"], "R_L": ["GOV", "TRS"], "R_E": ["IND", "MAT"],          # Table 12.2
                      "strategic-risk ceiling": ["IND", "DIG", "MAT"], "closure buffer B": ["TRS"]},
             "reads": {k: CONTRACTS[k] for k in ("liquidity", "strategic risk")},
             "buffers": {"closure buffer B": ("cash above the floor A", BUFFER)}},
}

def compatibility(U):
    # (C1)-(C5) on the layers attached so far (Definition 13.4)
    L = U["layers"].values()
    c1 = all(c in U["components"] for d in L for refs in d["refs"].values() for c in refs)
    contracts = {}
    for d in L:
        for ind, con in d.get("reads", {}).items():
            contracts.setdefault(ind, set()).add(con)
    c2 = all(len(v) == 1 for v in contracts.values())
    stocks = {s for d in L for s in d.get("stocks", {})}
    buffers = [b for d in L for b in d.get("buffers", {}).values()]
    c3 = all(fund in stocks for fund, _ in buffers)
    claims = sum(v for d in L for v in d.get("claims", {}).values()) + sum(b for fund, b in buffers if fund in stocks)
    c4 = claims <= POOL                # Table 13.3: for Meridian, (C4) is the joint check of the claims on treasury cash
    has_build = any("digital build: write set" in d["refs"] for d in L)
    c5 = (c4 and CAPABILITY >= CAPABILITY_FLOOR) if has_build else None    # the build's domain against the joint region
    return {"C1": c1, "C2": c2, "C3": c3, "C4": c4, "C5": c5, "claims": claims}

def assemble(order):
    U = {"components": tuple(COMPONENTS), "arrows": tuple(ARROWS), "layers": {}}
    print("attach in the order " + ", ".join(order) + ":")
    for name in order:
        U["layers"][name] = LAYERS[name]                       # integration adjoins a declaration (Definition 13.3)
        c = compatibility(U)
        flags = "  ".join(f"{k} {'pass' if c[k] else 'FAIL'}" for k in ("C1", "C2", "C3", "C4"))
        c5 = "no operator attached yet" if c["C5"] is None else f"the build's domain {'nonempty' if c['C5'] else 'empty'}"
        print(f"  + {name:15s} {flags}   claims on treasury cash {c['claims']:6.1f} of {POOL:.0f}   C5: {c5}")
    return U, c

U1, last1 = assemble(["transformation", "capital", "performance", "risk"])
U2, last2 = assemble(["capital", "risk", "performance", "transformation"])
print("the two orders give the same unified architecture:", U1 == U2, "  and the same checks:", last1 == last2)
print("each layer returned unchanged on restriction:", all(U1["layers"][k] is LAYERS[k] for k in LAYERS))
print(f"(C4) fails by {last1['claims'] - POOL:.1f}: the joint check of Worked Example 13.1 (Panel 5)")

shared = {}
for name, d in LAYERS.items():
    for ind, con in d.get("reads", {}).items():
        shared.setdefault(ind, []).append(name)
for ind, who in shared.items():
    if len(who) > 1:
        print(f"(C2) {ind} is read by {', '.join(who)}: one contract, {CONTRACTS[ind]}")

R = np.array([[1 if (a, b) in {(x, y) for x, y, _ in ARROWS} else 0 for b in COMPONENTS] for a in COMPONENTS])
closure = np.linalg.matrix_power(R + np.eye(7, dtype=int), 6) > 0             # who reaches whom
core = [c for i, c in enumerate(COMPONENTS) if any(closure[i, j] and closure[j, i] for j in range(7) if j != i)]
RENT = 0.025 * 58 * 4.0 * 4                                                     # Chapter 10: delta x 58 at $4m a point
print("\nTable 13.1, Meridian's instance:")
print(f"  state           {len(COMPONENTS)} components; a feedback core of {len(core)}: {', '.join(core)}")
print(f"  transformation  the digital build writes to Digital and engineering; deepest draw ${DRAW:.1f} million")
print(f"  capital         {len(FORMS)} forms; a capability rent of 0.025 x 58 = 1.45 points a quarter at $4 million: ${RENT:.1f} million a year")
print(f"  performance     {len(DIMENSIONS)} dimensions; the CFO's scorecard reads liquidity, technology, efficiency and strategic risk")
print(f"  risk            ${POOL:.0f} million of armor above the floor; the closure buffer, ${BUFFER:.1f} million, is treasury cash")

def var_es(values, probs, alpha=0.95):
    # value at risk and expected shortfall of a discrete loss law (Chapter 12)
    o = np.argsort(values); v, p = np.asarray(values, float)[o], np.asarray(probs, float)[o]
    q = v[np.searchsorted(np.cumsum(p), alpha - 1e-15)]
    return q, (p[v > q] @ v[v > q] + q * (p[v <= q].sum() - alpha)) / (1 - alpha)
one, pair = var_es([0, 100], [0.96, 0.04]), var_es([0, 100, 200], [0.96 ** 2, 2 * 0.04 * 0.96, 0.04 ** 2])
print("\nnon-redundancy (Theorem 13.1(iv)): two loans that each lose 100 with probability 0.04, consolidated")
print(f"  risk layer declaring VaR at 95%: {one[0]:.0f} + {one[0]:.0f} apart, {pair[0]:.0f} together -> consolidating raises measured risk")
print(f"  risk layer declaring ES at 95%:  {one[1]:.0f} + {one[1]:.0f} apart, {pair[1]:.1f} together -> it lowers it")

## Panel 2 — Wire the couplings (Theorem 13.3(ii); Figure 13.2; Online Appendix 13.B.1: 2, 4, 10, unbounded)

AXIOM Lab 1.13, step 2. Losses propagate on the layered graph with the gains of the interaction matrix
$\mathbf{M}$, rows receiving: $\boldsymbol{\ell}_{k+1} = \mathbf{M}\boldsymbol{\ell}_k$, and the total impact
of a unit loss, summed over layers and periods, is finite exactly when $\rho(\mathbf{M}) < 1$. Figure 13.2's
three layers each keep half of a quarter's loss ($a = 0.5$): alone, each books a unit shock
$1/(1 - a) = 2$ times over and passes its own stress test. Let a share $c$ cross from capital to performance,
performance to risk, and risk to capital. Every column of $\mathbf{M}$ then sums to $a + c$, so
$\rho(\mathbf{M}) = a + c$ and the total impact is $1/(1 - a - c)$: **4** at $c = 0.25$, **10** at
$c = 0.40$, and from $c = 0.5$ on no finite bill, while each layer alone still reports 2. The spectral report
splits $\rho(\mathbf{M})$ by block, giving each block its eigenvalue-sensitivity weight
$\mathbf{u}_\ell^{\top}\mathbf{M}_{\ell\ell'}\mathbf{v}_{\ell'}/\mathbf{u}^{\top}\mathbf{v}$ ($\mathbf{u}$,
$\mathbf{v}$ the left and right Perron vectors): the diagonal blocks carry exactly $a$, the layers' own radius,
and the cross-layer blocks carry $c$, the whole difference (Theorem 13.3(ii)). To build a case in which every
layer is stable and the whole is not, take Online Appendix 13.B.1's construction: layers as stable as one
likes ($a = 0.05$) and $c = 1$.

In [ ]:
LAYERS3 = ["capital", "performance", "risk"]

def cycle_matrix(c, a=0.5):
    # capital -> performance -> risk -> capital; rows receive, columns send (Online Appendix 13.B.1)
    return np.array([[a, 0.0, c], [c, a, 0.0], [0.0, c, a]])

def rho(M):
    return float(max(abs(np.linalg.eigvals(M))))

def total_impact(M, seed=0):
    # sum over layers and periods of a unit loss seeded in one layer, exact transmission: 1'(I - M)^-1 e
    if rho(M) >= 1 - 1e-12:
        return np.inf                                  # no finite bill
    return float(np.linalg.solve(np.eye(len(M)) - M, np.eye(len(M))[:, seed]).sum())

def spectral_report(M, label):
    w, V = np.linalg.eig(M); v = np.abs(V[:, np.argmax(w.real)].real)
    w2, U = np.linalg.eig(M.T); u = np.abs(U[:, np.argmax(w2.real)].real)
    r = rho(M); share = np.outer(u, v) * M / (u @ v)          # share[i, j] * r: block (i, j)'s part of rho(M)
    diag, cross = float(np.trace(share)), float(share.sum() - np.trace(share))
    radii = ", ".join(f"{M[i, i]:.2f}" for i in range(len(M)))
    bill = total_impact(M)
    print(f"{label:34s} layer radii {radii}   rho(M) {r:.2f} = diagonal blocks {diag:.2f} + cross-layer blocks {cross:.2f}"
          f"   total impact {'%.1f' % bill if np.isfinite(bill) else 'unbounded'}")
    return r, diag, cross

print("each layer alone: radius 0.5, total impact 1/(1 - 0.5) =", 1 / (1 - 0.5))
GRID = (0.0, 0.25, 0.40, 0.45, 0.50)
REPORT = {c: spectral_report(cycle_matrix(c), f"Figure 13.2, c = {c:.2f}") for c in GRID}
BILL = {c: total_impact(cycle_matrix(c)) for c in GRID}
spectral_report(cycle_matrix(1.0, a=0.05), "every layer at 0.05, c = 1")
print("column sums at c = 0.25:", cycle_matrix(0.25).sum(axis=0), "-> rho(M) = a + c exactly (Online Appendix 13.B.1)")

Q = 40
fig, (ax, bx) = plt.subplots(1, 2, figsize=(7.6, 3.6), gridspec_kw={"width_ratios": [1.15, 1]})
for c, col, ls in ((0.0, "#616161", ":"), (0.25, "#1F4E79", "-"), (0.40, "#E65100", "-"), (0.50, "#B71C1C", "--")):
    M, loss, cum = cycle_matrix(c), np.array([1.0, 0.0, 0.0]), [1.0]
    for _ in range(Q):
        loss = M @ loss; cum.append(cum[-1] + loss.sum())
    lab = "each layer alone: 2" if c == 0 else (f"c = {c:.2f}: " + ("no finite bill" if c >= 0.5 else f"{BILL[c]:.0f}"))
    ax.plot(range(Q + 1), cum, color=col, lw=1.8, ls=ls, label=lab)
ax.set(xlabel="quarters after a unit loss", ylabel="cumulative bill, all layers (× the shock)", xlim=(0, Q), ylim=(0, 22),
       title="The bill quarter by quarter")
ax.legend(frameon=False, fontsize=7, loc="upper left")
cs = np.linspace(0, 0.4985, 400)
bx.axvspan(0.5, 0.62, color="#F6E3E3", lw=0)
bx.plot(cs, [1 / (1 - 0.5 - c) for c in cs], color="#B71C1C", lw=1.8)
bx.axhline(2, color="#616161", lw=0.9, ls="--")
bx.text(0.02, 2.4, "each layer alone: 2", fontsize=7, color="#616161")
for c in (0.25, 0.40):
    bx.plot([c], [BILL[c]], "o", color="#B71C1C", ms=4)
    bx.text(c - 0.015, BILL[c] + 0.3, f"{BILL[c]:.0f}×", fontsize=7, color="#B71C1C", ha="right")
bx.text(0.56, 8, "no finite\nbill", fontsize=7, color="#B71C1C", ha="center")
bx.set(xlabel="cross-layer share, c", ylabel="total impact of a unit shock", xlim=(0, 0.62), ylim=(0, 12),
       title="Coupled, the bill is 1/(1 − a − c)")
for z in (ax, bx):
    z.tick_params(labelsize=7); z.xaxis.label.set_size(8); z.yaxis.label.set_size(8); z.title.set_size(9)
fig.suptitle("Every layer passes its own test, the whole fails (seed 26113)", fontsize=10)
plt.tight_layout(); plt.show()

## Panel 3 — Follow the wavefront (Proposition 13.2; Theorem 13.4(iii); Online Appendix 13.B.3: treasury and Industrial Systems after one quarter; Materials, the market and governance never)

AXIOM Lab 1.13, step 3. Seed the digital program at its write set, Digital and engineering, and a demand
shock at the market, in the same quarter. A readout moves no sooner than the distance from the seed to its
support, and never if its support cannot be reached (Proposition 13.2; Theorem 13.4(iii)), so each readout's
**ledger clock** is that distance. Treasury is one arrow from Digital and Industrial Systems one arrow from
engineering: liquidity, the covenant and Industrial Systems' operations can respond to the program after one
quarter. Advanced Materials receives arrows only from the market, and the market and governance receive none:
no readout supported only there can ever respond to the program, so crediting Materials' results to it is
refuted by the wiring. The demand shock reaches the three divisions after one quarter and engineering and
treasury after two (Worked Example 9.1). The readouts and their supports are the book's: capital (Table 9.1),
performance (Table 11.4), risk (Tables 9.3 and 12.2).

In [ ]:
WRITE = ["DIG", "ENG"]                                     # the digital program's write set
PROGRAM, SHOCK = reach(WRITE), reach(["MKT"])
fmt = lambda d: "never" if d is None else str(d)
print("first quarter in which a change can reach each component")
print(f"  {'component':26s} {'digital program':>16s} {'demand shock':>13s}")
for c in COMPONENTS:
    print(f"  {NAMES[c] + ' (' + c + ')':26s} {fmt(PROGRAM[c]):>16s} {fmt(SHOCK[c]):>13s}")
for name, dist in (("digital program", PROGRAM), ("demand shock", SHOCK)):
    waves = {}
    for c, d in dist.items():
        waves.setdefault(d, []).append(c)
    print(f"the {name}'s wavefront: " + "; ".join(f"quarter {d}: {', '.join(waves[d])}" for d in sorted(k for k in waves if k is not None))
          + (f"; never: {', '.join(waves[None])}" if None in waves else ""))

READOUTS = [("capital", "liquidity (financial capital)", ["TRS"]), ("capital", "net leverage (financial capital)", ["TRS"]),
            ("capital", "workforce capability (human capital)", ["ENG"]),
            ("performance", "P_F financial", ["TRS"]), ("performance", "P_O operational", ["IND", "MAT"]),
            ("performance", "P_S strategic", ["IND", "DIG", "MAT"]), ("performance", "P_T technological", ["DIG"]),
            ("performance", "P_I innovation", ["DIG", "ENG"]), ("performance", "P_R resilience", ["TRS"]),
            ("risk", "R_F financial: floor, covenant", ["TRS"]), ("risk", "R_O operational", ["IND", "MAT"]),
            ("risk", "R_S strategic", ["MKT", "GOV"]), ("risk", "R_T technological", ["DIG", "ENG"]),
            ("risk", "R_G governance", ["GOV"]), ("risk", "R_L legal and regulatory", ["GOV", "TRS"]),
            ("risk", "R_E environmental", ["IND", "MAT"]), ("risk", "strategic-risk ceiling", ["IND", "DIG", "MAT"])]

def clock(support, dist):
    hit = [dist[c] for c in support if dist[c] is not None]
    return min(hit) if hit else None

print("\nthe ledger clocks: first quarter in which each readout can respond")
print(f"  {'ledger':12s} {'readout':36s} {'support':14s} {'program':>8s} {'shock':>6s}")
CLOCKS = []
for ledger, name, support in READOUTS:
    p, s = clock(support, PROGRAM), clock(support, SHOCK)
    CLOCKS.append((ledger, name, support, p, s))
    print(f"  {ledger:12s} {name:36s} {'+'.join(support):14s} {fmt(p):>8s} {fmt(s):>6s}")
never = [c for c in COMPONENTS if PROGRAM[c] is None]
print(f"\nthe program never reaches {', '.join(NAMES[c] for c in never)}: {len(never)} of {len(COMPONENTS)} components;")
print(f"Industrial Systems is {reach(['DIG'])['IND']} quarters from Digital alone (Table 11.4's floor for P_T -> P_O), "
      f"{PROGRAM['IND']} from the program, which also writes to engineering")
for seed, j in (("the digital program", 3), ("the demand shock", 4)):
    first = {lg: min(r[j] for r in CLOCKS if r[0] == lg and r[j] is not None) for lg in ("capital", "performance", "risk")}
    print(f"first quarter in which each ledger can respond to {seed}: " + ", ".join(f"{lg} {q}" for lg, q in first.items()))
print("every ledger with a readout on the write set responds at once (Proposition 13.2); each readout on its own clock")

## Panel 4 — Read the sign-offs (Worked Example 13.1, Step 1; Online Appendix 13.B.4: \$138 million at quarter 7, \$187 million, \$132 and \$83 million to spare)

AXIOM Lab 1.13, step 4, first part: load the funding, buffer and schedule sign-offs. Liquidity today is
\$620 million and the board's floor \$350 million, so \$270 million lies above the floor. On Chapter 8's plan
the program's deepest draw is **\$138 million, at quarter 7**: funded, with \$132 million to spare. The risk
office sizes its buffer against a closed funding window at the 90th-percentile closure of Chapter 12,
**\$187 million**: funded, with \$83 million to spare. The schedule sign-off accepts the unphased plan because
the buffer is held; it involves no constraint between the draw and the buffer alone. Each check is correct.

In [ ]:
PLAN, PHASED = plan(DEPTH), plan(DEPTH_PHASED)
K7 = int(PLAN.argmin())
MEDIAN = stats.gamma.ppf(0.50, SHAPE, scale=SCALE)
print("quarter      " + "".join(f"{k:6d}" for k in range(13)))
print("plan         " + "".join(f"{v:6.0f}" for v in PLAN))
print("phased build " + "".join(f"{v:6.0f}" for v in PHASED))
print(f"\nthe program's deepest draw: {LIQ0:.0f} - {PLAN.min():.1f} = {DRAW:.1f} at quarter {K7}")
print(f"closures (gamma, mean 120, sd 50): median {MEDIAN:.1f}, 90th percentile {BUFFER:.1f}")
print(f"funding sign-off   (CFO and transformation office): draw   {DRAW:6.1f} <= {POOL:.0f}: {DRAW <= POOL}, {POOL - DRAW:.1f} to spare")
print(f"buffer sign-off    (CFO and CRO):                   buffer {BUFFER:6.1f} <= {POOL:.0f}: {BUFFER <= POOL}, {POOL - BUFFER:.1f} to spare")
print(f"schedule sign-off  (transformation office and CRO): the unphased plan, because the buffer is held: no constraint between the two alone")

## Panel 5 — Walk the cycle as one constraint (Worked Example 13.1, Steps 2–3; Theorem 13.5(ii)–(iii), (v); Online Appendix 13.B.2; Figure 13.3: \$325 million on \$270 million)

AXIOM Lab 1.13, step 4, second part. Both claims rest on one asset, treasury's cash above the floor. The
draw $D$ is a transformation node at Digital that flows through the cash arrow to treasury; the buffer $B$ is a
risk node at treasury, a claim on that cash by its funding identity (C3); the cash $A$ is a capital node at
treasury. The constraint that matters, $D + B \le A$, holds all three in its scope, so its constraint graph is
a triangle — a cycle — and by Theorem 13.5(iii) no bilateral sign-off can certify it: each pair is
satisfiable while the three together are not (Theorem 13.5(ii)). The joint check: $138 + 187 = 325$ against
\$270 million, **infeasible by \$55 million**. On the plan, liquidity runs inside the buffer line
$350 + 187 = 537$ **from quarter 3 to the end of the program, 55 deep at quarter 7**; the phased build only in
quarters 5–8, by at most 6. When a tripwire fires, the error lies among the ancestors of its scope
(Theorem 13.5(v)): every component reaches treasury, so an invariant failing on treasury's books clears none
of them, while one failing on Materials' books clears all but the market and Materials.

In [ ]:
NODES = {"D": ("transformation", "DIG", DRAW), "B": ("risk", "TRS", BUFFER), "A": ("capital", "TRS", POOL)}
SCOPE = ["D", "B", "A"]                                         # the one constraint D + B <= A
EDGES = {(SCOPE[i], SCOPE[j]) for i in range(3) for j in range(i + 1, 3)}
print(f"constraint graph: nodes {SCOPE}, edges {sorted(EDGES)}; independent cycles {len(EDGES) - len(SCOPE) + 1}: a triangle")
D, B, A = DRAW, BUFFER, POOL
print("bilateral sign-offs, each a projection of D + B <= A:")
print(f"  funding  (D, A): {D:.1f} <= {A:.0f}   {'pass' if D <= A else 'fail'}")
print(f"  buffer   (B, A): {B:.1f} <= {A:.0f}   {'pass' if B <= A else 'fail'}")
print(f"  schedule (D, B): no constraint on the pair alone   pass")
JOINT = A - (D + B)
print(f"joint check: {D:.1f} + {B:.1f} = {D + B:.1f} against {A:.0f}: {'feasible' if JOINT >= 0 else 'infeasible'} by {abs(JOINT):.1f}")

BUFFER_LINE = FLOOR + BUFFER
inside = [k for k in range(13) if PLAN[k] < BUFFER_LINE]
inside_ph = [k for k in range(13) if PHASED[k] < BUFFER_LINE]
print(f"\nbuffer line: {FLOOR:.0f} + {BUFFER:.1f} = {BUFFER_LINE:.1f}")
print(f"the plan runs inside it in quarters {inside[0]}-{inside[-1]} ({len(inside)} quarters), deepest "
      f"{BUFFER_LINE - PLAN.min():.1f} at quarter {K7}")
print(f"the phased build runs inside it in quarters {inside_ph[0]}-{inside_ph[-1]}, by at most {BUFFER_LINE - PHASED.min():.1f}")

ancestors = lambda v: [c for c in COMPONENTS if reach([c])[v] is not None]
for v, books in (("TRS", "treasury's books"), ("MAT", "Materials' books")):
    anc = ancestors(v)
    cleared = [c for c in COMPONENTS if c not in anc]
    print(f"a tripwire on {books}: suspects {', '.join(anc)}; cleared: {', '.join(cleared) if cleared else 'none'}")

ROWS = [("As signed off", DRAW, BUFFER), ("Buffer at the median closure", DRAW, MEDIAN),
        ("Phased build", LIQ0 - PHASED.min(), BUFFER), ("Standby line", DRAW, max(BUFFER - LINE, 0.0)),
        ("Both levers", LIQ0 - PHASED.min(), max(BUFFER - LINE, 0.0))]
fig, (ax, bx) = plt.subplots(1, 2, figsize=(7.6, 3.6), gridspec_kw={"width_ratios": [1.1, 1]})
q = np.arange(13)
ax.fill_between(q, PLAN, BUFFER_LINE, where=PLAN < BUFFER_LINE, interpolate=True, color="#F3D9D9", lw=0)
ax.axhline(FLOOR, color="#616161", lw=0.9)
ax.axhline(BUFFER_LINE, color="#B71C1C", lw=0.9, ls="--")
ax.plot(q, PLAN, color="#1F4E79", lw=1.8, marker="o", ms=3, label="plan")
ax.plot(q, PHASED, color="#2E7D32", lw=1.4, ls="--", label="phased build")
ax.text(12.2, FLOOR + 5, "floor 350", fontsize=7, color="#616161", ha="right", va="bottom")
ax.text(11.6, BUFFER_LINE - 4, f"buffer line {BUFFER_LINE:.0f}", fontsize=7, color="#B71C1C", ha="right", va="top")
ax.text(7, 445, f"cash promised twice:\n{BUFFER_LINE - PLAN.min():.0f} deep at quarter {K7}", fontsize=7, color="#B71C1C", ha="center")
ax.set(xlabel="quarter of the program", ylabel="liquidity ($ million)", xlim=(0, 12.4), ylim=(300, 640), xticks=range(0, 13, 2),
       title="One pool, two claims")
ax.legend(frameon=False, fontsize=7, loc="upper right", bbox_to_anchor=(1.0, 0.93))
y = np.arange(len(ROWS))[::-1]
for yy, (name, d, b) in zip(y, ROWS):
    bx.barh(yy, d, height=0.56, color="#5E35B1", lw=0)
    bx.barh(yy, b, left=d, height=0.56, color="#B71C1C", alpha=0.85, lw=0)
    spare = POOL - d - b
    bx.text(max(d + b, POOL) + 6, yy, f"{spare:+.0f}", fontsize=7.5, va="center", fontweight="bold",
            color="#B71C1C" if spare < 0 else "#2E7D32")
bx.axvline(POOL, color="#2E7D32", lw=1.1)
bx.text(POOL - 4, len(ROWS) - 0.45, "pool 270", fontsize=7, color="#2E7D32", ha="right")
bx.text(8, -0.8, "draw", fontsize=7, color="#5E35B1", fontweight="bold", va="center")
bx.text(70, -0.8, "cash buffer", fontsize=7, color="#B71C1C", fontweight="bold", va="center")
bx.set_yticks(y); bx.set_yticklabels(["As signed off", "Median buffer", "Phased build", "Standby line", "Both levers"])
bx.set(xlabel="claims on the cash above the floor ($ million)", xlim=(0, 380), ylim=(-1.1, len(ROWS) - 0.2), title="The joint check")
for z in (ax, bx):
    z.tick_params(labelsize=7); z.xaxis.label.set_size(8); z.yaxis.label.set_size(8); z.title.set_size(9)
fig.suptitle("The same cash is promised twice: 325 of claims on 270 (seed 26113)", fontsize=10)
plt.tight_layout(); plt.show()

## Panel 6 — Repair it three ways (Worked Example 13.1, Step 4; Table 13.6; Online Appendix 13.B.4: +19, −6, +45, +94)

AXIOM Lab 1.13, step 4, last part. Each repair lives in its own ledger. Relaxing the buffer to the **median
closure**, \$113 million, a risk-ledger repair, passes with \$19 million to spare, but only by lowering the
bar: half of all closures would exceed it, and since the declaration does not change the cash, the breach
odds stay at 24.6 percent. **Phasing the build**, a transformation repair, cuts the draw to \$89 million but
alone still fails, by \$6 million. The **standby line** of Chapter 8, a capital repair, covers \$100 million
of any closure, cuts the cash buffer to \$87 million and passes with \$45 million to spare. **Both levers**
pass with \$94 million. One million fresh closures drawn with the lab's seed confirm the two quantiles the
buffers rest on.

In [ ]:
print(f"Table 13.6 ($ million)        {'draw':>6s} {'cash buffer':>12s} {'total claims':>13s} {'spare cash':>11s}")
for name, d, b in ROWS:
    print(f"  {name:28s} {d:6.1f} {b:12.1f} {d + b:13.1f} {POOL - d - b:+11.1f}   {'passes' if d + b <= POOL else 'fails'}")
TABLE = {name: (d, b, d + b, POOL - d - b) for name, d, b in ROWS}
print(f"the phased build's deepest draw: {LIQ0:.0f} - {PHASED.min():.1f} = {LIQ0 - PHASED.min():.1f} (quarter {int(PHASED.argmin())})")
print(f"the standby line covers {LINE:.0f} of a closure: cash buffer {BUFFER:.1f} - {LINE:.0f} = {BUFFER - LINE:.1f}; "
      f"cash and line together still cover the 90th percentile")

rng = np.random.default_rng(SEED)                       # fresh draws, the lab's seed
sizes = rng.gamma(SHAPE, SCALE, 1_000_000)
print(f"\none million fresh closures (seed {SEED}): median {np.median(sizes):.1f} (exact {MEDIAN:.1f}), "
      f"90th percentile {np.percentile(sizes, 90):.1f} (exact {BUFFER:.1f})")
print(f"  share exceeding the median buffer {100 * (sizes > MEDIAN).mean():.1f}%: half of all closures; exceeding the "
      f"90th-percentile cover {100 * (sizes > BUFFER).mean():.1f}%")

## Panel 7 — Run the program on one clock (Proposition 13.2; Theorem 13.4; Figure 13.1: months 11, 21 and 29)

AXIOM Lab 1.13, step 5, first part: run the program through all ledgers on one clock. Chapter 1's model,
month by month, read in three ledgers: capital (liquidity against the \$350 million floor), performance (the
CFO's scorecard against today's 37.0) and risk (strategic risk against its ceiling of 75, which the risk
committee measures to ±6 points). The scorecard and strategic risk are at their worst in **month 11**
(32.1, and 73.55, a ceiling 1.4 points away), liquidity in **month 21** (\$482 million), and the scorecard
regains today's level only in **month 29** (37.19). A review at month 11 in any one ledger reaches a verdict
that is correct in its ledger and unsafe for the program: performance sees a failing program, liquidity a
falling but ample cushion, risk a ceiling 1.4 points away, inside the instrument's error.

In [ ]:
MONTHS = np.arange(61)
LED = digital(MONTHS / 12)
LIQ_M, SCORE_M, RISK_M = PER_POINT * LED["L"], score(LED), LED["R"]
TODAY = SCORE_M[0]
i_liq, i_sc, i_rk = int(LIQ_M.argmin()), int(SCORE_M.argmin()), int(RISK_M.argmax())
i_back = next(m for m in range(i_sc, 61) if SCORE_M[m] >= TODAY)
print(f"capital:     liquidity low ${LIQ_M[i_liq]:.1f} million in month {i_liq}")
print(f"performance: the scorecard falls from {TODAY:.1f} to {SCORE_M[i_sc]:.2f} in month {i_sc} and regains today's level "
      f"in month {i_back} ({SCORE_M[i_back]:.2f}; {SCORE_M[i_back - 1]:.2f} in month {i_back - 1})")
print(f"risk:        strategic risk peaks at {RISK_M[i_rk]:.2f} in month {i_rk}, {75 - RISK_M[i_rk]:.1f} under the ceiling of 75 "
      f"(measured +/-6)")
m = 11
print(f"\nthe three reviews at month {m}, each in its own ledger:")
print(f"  performance: scorecard {SCORE_M[m]:.1f} against today's {TODAY:.1f} -> a failing program")
print(f"  capital:     liquidity ${LIQ_M[m]:.0f} million, falling ({LIQ_M[m] - LIQ_M[m - 1]:+.1f} a month), "
      f"${LIQ_M[m] - FLOOR:.0f} million above the floor -> an ample cushion")
print(f"  risk:        strategic risk {RISK_M[m]:.1f}, the ceiling {75 - RISK_M[m]:.1f} points away -> inside the +/-6 error")

fig, axes = plt.subplots(1, 3, figsize=(7.6, 3.6))
specs = [(LIQ_M, "Capital", "#2E7D32", "liquidity ($ million)", FLOOR, "floor 350", (300, 660), i_liq, "low"),
         (SCORE_M, "Performance", "#E65100", "CFO's scorecard", TODAY, "today 37.0", (28, 50), i_sc, "low"),
         (RISK_M, "Risk", "#B71C1C", "strategic risk (index)", 75.0, "ceiling 75", (42, 86), i_rk, "peak")]
for ax, (y, name, col, ylab, ref, reflab, ylim, i, word) in zip(axes, specs):
    if name == "Risk":
        ax.fill_between(MONTHS, y - 6, y + 6, color="#F3D9D9", lw=0)
    ax.axhline(ref, color="#616161" if name != "Risk" else "#B71C1C", lw=0.9, ls="--" if name == "Performance" else "-")
    ax.plot(MONTHS, y, color=col, lw=1.8)
    ax.plot([i], [y[i]], "o", color=col, ms=4)
    val = f"{y[i]:.0f}" if name == "Capital" else f"{y[i]:.1f}"
    ax.annotate(f"{word} {val},\nmonth {i}", (i, y[i]), xytext=(i + 4, y[i] + (-0.12 if word == "low" else 0.10) * (ylim[1] - ylim[0])),
                fontsize=7, color=col, va="center")
    ax.text(59, ref + 0.012 * (ylim[1] - ylim[0]), reflab, fontsize=7, color="#616161" if name != "Risk" else "#B71C1C",
            ha="right", va="bottom")
    if name == "Risk":
        ax.text(59, 44, "band: error ±6", fontsize=7, color="#616161", ha="right", va="bottom")
    for mm in (11, 21):
        ax.axvline(mm, color="#9E9E9E", lw=0.5, ls=":")
    ax.set(xlim=(0, 60), ylim=ylim, xticks=range(0, 61, 12), xlabel="month")
    ax.set_ylabel(ylab, fontsize=8); ax.set_title(name, fontsize=9, color=col, loc="left")
    ax.tick_params(labelsize=7); ax.xaxis.label.set_size(8)
axes[1].plot([i_back], [SCORE_M[i_back]], "o", color="#E65100", mfc="white", ms=4)
axes[1].annotate(f"today again,\nmonth {i_back}", (i_back, SCORE_M[i_back]), xytext=(i_back + 4, SCORE_M[i_back] - 3),
                 fontsize=7, color="#E65100", va="center")
fig.suptitle("One program, three ledgers, three clocks (seed 26113)", fontsize=10)
plt.tight_layout(); plt.show()

## Panel 8 — Decide in one place (Worked Example 13.1, Step 5; Proposition 13.5; Theorem 13.4(iv); Worked Example 12.1: 24.6, 13.1, 12.4 and 3.0 percent)

AXIOM Lab 1.13, step 5, second part: compare the separated verdicts with the unified one. Deciding alone, each
office guards its own ledger: the CRO will not shrink the buffer, the transformation office will not delay its
technology gains, and the CFO will not pay for a line it does not use. Each repair taken alone is a separated
choice, and the unified object reads every repair in every ledger: the joint check, and the risk ledger of
Worked Example 12.1, the breach odds over Chapter 8's 100,000 paths (**seed 8**, the book's stream) against
the 5 percent appetite. The joint check rules out the plan as signed off and the phased build alone; the risk
ledger rules out the smaller buffer and the line alone, at **24.6** and **12.4 percent**; only **both levers**
pass every ledger, at **3.0 percent**. A separated procedure is the unified problem with the other ledgers'
repairs ruled out, so it never does better, and here strictly worse: its extra constraints exclude the only
integrated answer (Proposition 13.5(ii)). The flaw of averages runs through every ledger at once
(Theorem 13.4(iv)): the plan's low point is \$482 million, while its simulated paths bottom out at
\$384 million on average. A fresh stream drawn with the lab's seed returns the same verdicts.

In [ ]:
CONFIGS = {"As signed off": {}, "Buffer at the median closure": {}, "Phased build": {"depth": DEPTH_PHASED},
           "Standby line": {"line": LINE}, "Both levers": {"line": LINE, "depth": DEPTH_PHASED}}
LEDGER = {"As signed off": "-", "Buffer at the median closure": "risk", "Phased build": "transformation",
          "Standby line": "capital", "Both levers": "transformation + capital"}     # Step 4: where each repair lives

def odds(L):
    lows = L[:, 1:].min(axis=1)
    p = float((lows < FLOOR).mean())
    return p, np.sqrt(p * (1 - p) / len(lows)), lows

BOOK, FRESH, _runs = {}, {}, {}
for name, kw in CONFIGS.items():                         # a buffer declaration does not change the cash: same paths
    key = tuple(sorted(kw.items()))
    if key not in _runs:
        _runs[key] = (odds(simulate(**kw)), odds(simulate(seed=SEED, **kw)))   # the book's paths (seed 8); a fresh stream
    BOOK[name], FRESH[name] = _runs[key]
LOWS = BOOK["As signed off"][2]
print(f"{'declaration':30s} {'ledger of the repair':24s} {'joint check':>12s} {'breach (seed 8)':>16s} {'risk ledger':>12s} {'every ledger':>13s}")
VERDICT = {}
for name in CONFIGS:
    spare, (p, se, _) = TABLE[name][3], BOOK[name]
    joint_ok, risk_ok = spare >= 0, p <= APPETITE
    VERDICT[name] = joint_ok and risk_ok
    print(f"{name:30s} {LEDGER[name]:24s} {spare:+7.1f} {'pass' if joint_ok else 'fail':>4s} {100 * p:9.1f}% ± {100 * se:.2f}"
          f" {'pass' if risk_ok else 'fail':>9s} {'yes' if VERDICT[name] else 'no':>13s}")
separated = ["Buffer at the median closure", "Phased build", "Standby line"]
print(f"\nseparated choices (one ledger's repair each): {len(separated)}; passing every ledger: {sum(VERDICT[s] for s in separated)}")
print(f"unified choices (the repairs and their combination): passing every ledger: {sum(VERDICT.values())}, "
      f"{', '.join(n for n in CONFIGS if VERDICT[n])}")
print(f"\nthe flaw of averages (Theorem 13.4(iv)): planned low ${PLAN.min():.1f} million; average of the simulated lows "
      f"${LOWS.mean():.1f} million (seed 8), {PLAN.min() - LOWS.mean():.0f} below the plan")
print("\nfresh stream (seed 26113), 100,000 paths each: the same verdicts")
for name in CONFIGS:
    (pb, sb, _), (pf, sf, _) = BOOK[name], FRESH[name]
    print(f"  {name:30s} {100 * pf:5.1f}% ± {100 * sf:.2f}  ({(pf - pb) / sb:+.1f} standard errors from the book's {100 * pb:.1f}%)"
          f"  risk ledger {'pass' if pf <= APPETITE else 'fail'}")

fig, ax = plt.subplots(figsize=(7.6, 3.6))
ax.add_patch(plt.Rectangle((0, 0), 130, 100 * APPETITE, color="#E3F0E4", lw=0, zorder=0))
ax.axvline(0, color="#616161", lw=0.9); ax.axhline(100 * APPETITE, color="#616161", lw=0.9, ls="--")
ax.text(127, 5.4, "appetite 5%", fontsize=7, color="#616161", ha="right")
ax.text(3, 0.7, "passes every ledger", fontsize=7, color="#2E7D32", ha="left")
ax.text(-3, 29, "← joint check fails", fontsize=7, color="#616161", ha="right")
ax.text(3, 29, "joint check passes →", fontsize=7, color="#616161", ha="left")
place = {"As signed off": (0, 1.6, "center"), "Buffer at the median closure": (0, 1.6, "center"),
         "Phased build": (-5, 0, "right"), "Standby line": (9, 0, "left"), "Both levers": (9, 0, "left")}
for name, col in zip(CONFIGS, ("#B71C1C", "#B71C1C", "#5E35B1", "#2E7D32", "#1F4E79")):
    x, (pb, sb, _), (pf, sf, _) = TABLE[name][3], BOOK[name], FRESH[name]
    ax.plot([x], [100 * pb], "o", color=col, ms=6, zorder=3)
    ax.errorbar([x + 3.5], [100 * pf], yerr=[200 * sf], fmt="o", mfc="white", color=col, ms=5, capsize=2, lw=1, zorder=3)
    dx, dy, ha = place[name]
    ax.text(x + dx, 100 * pb + dy, name.replace("Buffer at the median closure", "Median buffer"), fontsize=7, color=col,
            ha=ha, va="center" if dy == 0 else "bottom")
ax.plot([], [], "o", color="#616161", ms=6, label="the book's stream (seed 8)")
ax.errorbar([], [], yerr=[], fmt="o", mfc="white", color="#616161", ms=5, capsize=2, lw=1, label="fresh stream (seed 26113), ± 2 s.e.")
ax.set(xlabel="spare cash in the joint check ($ million)", ylabel="breach probability (%)", xlim=(-80, 130), ylim=(0, 31),
       title="Only both levers pass every ledger (seed 26113)")
ax.legend(frameon=False, fontsize=7, loc="center right", bbox_to_anchor=(1.0, 0.62))
plt.tight_layout(); plt.show()

## Validation — the book's numbers, and agreement with `DCT_V1_Ch13_Lab.xlsx`

Each checkpoint is a number printed in Chapter 13 (Worked Example 13.1, Tables 13.1 and 13.6, Figures
13.1–13.3, Sections 13.5 and 13.6) or in Online Appendix 13.B.1–13.B.4. The breach odds and the average low are
the book's simulated paths (seed 8). The workbook recomputes the ones marked *workbook* with Excel formulas and
shows PASS against the same engine values on its `Checks` sheet.

In [ ]:
T = TABLE
checks = [
    ("components in the skeleton (workbook)", len(COMPONENTS), 7, 0),
    ("feedback core: components (workbook)", len(core), 4, 0),
    ("forms of capital (workbook)", len(FORMS), 10, 0),
    ("performance dimensions (workbook)", len(DIMENSIONS), 10, 0),
    ("capability rent, $ million a year (workbook)", RENT, 23, 0.5),
    ("cash above the floor today (workbook)", POOL, 270, 0),
    ("each layer alone: total impact (workbook)", 1 / (1 - 0.5), 2, 1e-9),
    ("total impact, no coupling (workbook)", BILL[0.0], 2, 1e-9),
    ("total impact at c = 0.25 (workbook)", BILL[0.25], 4, 1e-9),
    ("total impact at c = 0.40 (workbook)", BILL[0.40], 10, 1e-9),
    ("rho(M) at c = 0.25, a + c (workbook)", REPORT[0.25][0], 0.75, 1e-9),
    ("rho(M) at c = 0.5: no finite bill (workbook)", REPORT[0.50][0], 1.0, 1e-9),
    ("program: quarters to treasury (workbook)", PROGRAM["TRS"], 1, 0),
    ("program: quarters to Industrial Systems (workbook)", PROGRAM["IND"], 1, 0),
    ("program: components never reached (workbook)", len(never), 3, 0),
    ("demand shock: quarters to the divisions (workbook)", max(SHOCK[c] for c in ("IND", "DIG", "MAT")), 1, 0),
    ("demand shock: quarters to engineering (workbook)", SHOCK["ENG"], 2, 0),
    ("demand shock: quarters to treasury (workbook)", SHOCK["TRS"], 2, 0),
    ("plan liquidity, quarter 1 (workbook)", PLAN[1], 574, 0.5),
    ("plan liquidity, quarter 12 (workbook)", PLAN[12], 510, 0.5),
    ("phased liquidity, quarter 1 (workbook)", PHASED[1], 589, 0.5),
    ("phased liquidity, quarter 12 (workbook)", PHASED[12], 556, 0.5),
    ("deepest draw (workbook)", DRAW, 138, 0.5),
    ("quarter of the deepest draw (workbook)", K7, 7, 0),
    ("funding check: spare (workbook)", POOL - DRAW, 132, 0.5),
    ("closure: 90th percentile (workbook)", BUFFER, 187, 0.5),
    ("buffer check: spare (workbook)", POOL - BUFFER, 83, 0.5),
    ("closure: median (workbook)", MEDIAN, 113, 0.5),
    ("joint check: total claims (workbook)", T["As signed off"][2], 325, 0.5),
    ("joint check: spare (workbook)", T["As signed off"][3], -55, 0.5),
    ("buffer line (workbook)", BUFFER_LINE, 537, 0.5),
    ("plan inside the buffer line: first quarter (workbook)", inside[0], 3, 0),
    ("plan inside the buffer line: last quarter (workbook)", inside[-1], 12, 0),
    ("plan inside the buffer line: deepest (workbook)", BUFFER_LINE - PLAN.min(), 55, 0.5),
    ("phased inside the buffer line: first quarter (workbook)", inside_ph[0], 5, 0),
    ("phased inside the buffer line: last quarter (workbook)", inside_ph[-1], 8, 0),
    ("phased inside the buffer line: deepest (workbook)", BUFFER_LINE - PHASED.min(), 6, 0.5),
    ("median buffer: total claims (workbook)", T["Buffer at the median closure"][2], 251, 0.5),
    ("median buffer: spare (workbook)", T["Buffer at the median closure"][3], 19, 0.5),
    ("phased build: draw (workbook)", T["Phased build"][0], 89, 0.5),
    ("phased build: total claims (workbook)", T["Phased build"][2], 276, 0.5),
    ("phased build: spare (workbook)", T["Phased build"][3], -6, 0.5),
    ("standby line: cash buffer (workbook)", T["Standby line"][1], 87, 0.5),
    ("standby line: total claims (workbook)", T["Standby line"][2], 225, 0.5),
    ("standby line: spare (workbook)", T["Standby line"][3], 45, 0.5),
    ("both levers: total claims (workbook)", T["Both levers"][2], 176, 0.5),
    ("both levers: spare (workbook)", T["Both levers"][3], 94, 0.5),
    ("scorecard today (workbook)", TODAY, 37.0, 0.05),
    ("liquidity low, $ million (workbook)", LIQ_M[i_liq], 482, 0.5),
    ("month of the liquidity low (workbook)", i_liq, 21, 0),
    ("scorecard low (workbook)", SCORE_M[i_sc], 32.1, 0.05),
    ("month of the scorecard low (workbook)", i_sc, 11, 0),
    ("month the scorecard regains 37.0 (workbook)", i_back, 29, 0),
    ("scorecard in that month (workbook)", SCORE_M[i_back], 37.19, 0.005),
    ("strategic-risk peak (workbook)", RISK_M[i_rk], 73.55, 0.005),
    ("month of the risk peak (workbook)", i_rk, 11, 0),
    ("ceiling's distance at the peak (workbook)", 75 - RISK_M[i_rk], 1.4, 0.05),
    ("breach odds, plan as signed off, % (seed 8)", 100 * BOOK["As signed off"][0], 24.6, 0.05),
    ("breach odds, phased build, % (seed 8)", 100 * BOOK["Phased build"][0], 13.1, 0.05),
    ("breach odds, standby line, % (seed 8)", 100 * BOOK["Standby line"][0], 12.4, 0.05),
    ("breach odds, both levers, % (seed 8)", 100 * BOOK["Both levers"][0], 3.0, 0.05),
    ("planned low point, $ million (workbook)", PLAN.min(), 482, 0.5),
    ("average simulated low, $ million (seed 8)", LOWS.mean(), 384, 0.5),
    ("repairs that pass every ledger (workbook)", sum(VERDICT.values()), 1, 0),
]
ok, width = 0, max(len(name) for name, *_ in checks)
for name, got, want, tol in checks:
    status = "PASS" if abs(got - want) <= tol + 1e-9 else "FAIL"; ok += status == "PASS"
    print(f"{name:{width}s} {got:10.4f}   book {want:9.4f}   {status}")
print(f"\n{ok} of {len(checks)} checkpoints PASS")

**Next.** Chapter 14 asks what the assembled system does: well-posedness, equilibria, stability (opening with
the interaction matrix's spectral reading), sensitivity, and viability, whether the unified feasible region can
be maintained along a transformation path. AXIOM-13 runs the same model interactively (Lab 1.13); this notebook
and the workbook are its reproducible record.